# Screening Consolidation

Merges the three screening phases into one paper-level table with a single verdict per paper.

- **round1** — 50 papers, 3 annotators (20 seen by all three, 30 by exactly two), 20 disagreements resolved manually.
- **round2** — 20 papers, all 3 annotators saw all of them, 1 disagreement resolved manually.
- **remaining** — 116 papers, screened by a single annotator, no consolidation needed.

The three sets are disjoint and add up to the 186 deduplicated papers from
[02_data_merge.ipynb](02_data_merge.ipynb). Agreement statistics for the two IAA rounds are in
[03_iaa_analysis.ipynb](03_iaa_analysis.ipynb).

Output: `../data/processed/screening/screening_consolidated.xlsx`.

## Setup

`REMOVE_PHRASES` holds annotator commentary that is not a topic keyword. Every phrase is removed
from the `Topic/Ideas` text (case-insensitive, whole words only); a fragment left empty is dropped.
The list is meant to be edited by hand. 
The last section prints the long fragments that survived, so new candidates can be added and the notebook re-run.

In [14]:
import sys
sys.path.append("../")  # go up one level from notebooks/ to project root

import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

ANNOTATORS = ["MC", "ME", "TC"]

# Hierarchy used to break ties in the research-area majority vote. TC > ME > MC. 
TIE_BREAK = ["TC", "ME", "MC"]

# Paper-level columns, identical in the three source files, kept as the leading columns.
META_COLS = ["paper_id", "ids", "year", "sources", "title", "abstract",
             "published_in", "paper_type"]
OUT_COLS = META_COLS + ["include", "to_discuss", "research_area", "topic", "notes", "phase"]

RESOLUTION_MAP = {"include": True, "not include": False}

# Canonical research areas. Round 2 spells it "Polarazation"; "Both" stays its own category.
RESEARCH_AREA_CANON = {
    "misinformation": "Misinformation",
    "polarization": "Polarization",
    "polarazation": "Polarization",
    "both": "Both",
}

REMOVE_PHRASES = [
    # whole fragments that are commentary, not topics
    "Does not touch on polarization or misinformation",
    "This seems to be an interesting paper",
    "Interesting combination of methods",
    "Polariation is a particular parameter here",
    "not a phenomenon",
    "not enough information",
    "no climate",
    "No social media",
    "OT",
    "analysed through",
    "as a proxy for",
    "Notion of"
    # leading filler; the keyword that follows survives
    "too. Particularly the focus on",              # -> "technological optimism"
    "Potentially interesting",                     # -> "SNA approach"
    "Interesting to see how they operationalize",  # -> '"social noise"'
]

ROUND1_PATH = Path("../data/processed/iaa/annotated/round1/iaa_merged_resolution.xlsx")
ROUND2_PATH = Path("../data/processed/iaa/annotated/round2/iaa_merged_resolution.xlsx")
REMAINING_PATH = Path("../data/processed/screening/remaining_screening_annotated.xlsx")
OUTPUT_PATH = Path("../data/processed/screening/screening_consolidated.xlsx")

## Load the three phases

Round 1 also carries an `iaa_set` column between `paper_id` and `ids`; it is dropped, the `phase`
column records the provenance instead.

In [15]:
round1 = pd.read_excel(ROUND1_PATH)
round2 = pd.read_excel(ROUND2_PATH)
remaining = pd.read_excel(REMAINING_PATH)

assert round1.shape[0] == 50, f"round1 has {len(round1)} rows (expected 50)"
assert round2.shape[0] == 20, f"round2 has {len(round2)} rows (expected 20)"
assert remaining.shape[0] == 116, f"remaining has {len(remaining)} rows (expected 116)"

ids1, ids2, ids3 = (set(df["paper_id"]) for df in (round1, round2, remaining))
assert not (ids1 & ids2) and not (ids1 & ids3) and not (ids2 & ids3), "phases overlap"
assert len(ids1 | ids2 | ids3) == 186, "expected 186 unique papers"

for name, df in [("round1", round1), ("round2", round2), ("remaining", remaining)]:
    assert set(META_COLS) <= set(df.columns), f"{name} is missing metadata columns"
    print(f"{name:10} {df.shape[0]:>3} papers x {df.shape[1]:>2} columns")

round1      50 papers x 29 columns
round2      20 papers x 28 columns
remaining  116 papers x 13 columns


## Helpers

### `to_bool` — normalise the boolean columns

The three files disagree on how booleans were stored: round 1 has floats (`1.0` / `0.0` / blank,
blank meaning the annotator never saw the paper), round 2 has real booleans plus one stray
`'TRUE '` string, the remaining file has real booleans. Blank becomes `pd.NA`.

In [16]:
def to_bool(value):
    """Normalise a spreadsheet boolean to True / False / pd.NA (blank)."""
    if value is None or value is pd.NA or (not isinstance(value, str) and pd.isna(value)):
        return pd.NA
    if isinstance(value, str):
        text = value.strip().lower()
        if text == "":
            return pd.NA
        if text in {"true", "yes", "1"}:
            return True
        if text in {"false", "no", "0"}:
            return False
        raise ValueError(f"unrecognised boolean string: {value!r}")
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, float, np.integer, np.floating)):
        if value in (0, 1):
            return bool(value)
        raise ValueError(f"unrecognised boolean number: {value!r}")
    raise ValueError(f"unrecognised boolean value: {value!r} ({type(value)})")

### `consolidate_include` — one verdict per paper

Over the `*_Include` columns, treating a blank as "this annotator never saw the paper":

- every non-blank verdict `True` → **true** (so `(True, True, blank)` is a consensus),
- every non-blank verdict `False` → **false**,
- otherwise there is no consensus → take the manually filled `resolution`
  (`include` → `True`, `not include` → `False`).

Three invariants are enforced rather than assumed, so a later edit of the source files that leaves
a disagreement unresolved fails loudly instead of silently writing a blank verdict: every
disagreement has a resolution, every resolution string is recognised, and no consensus row carries
a stray resolution.

In [17]:
def consolidate_include(row):
    """True / False verdict for one paper from the per-annotator columns + `resolution`."""
    verdicts = [to_bool(row[f"{a}_Include"]) for a in ANNOTATORS]
    seen = [v for v in verdicts if v is not pd.NA]
    if not seen:
        raise ValueError(f"{row['paper_id']}: no annotator verdict at all")

    if all(seen):
        consensus = True
    elif not any(seen):
        consensus = False
    else:
        consensus = None  # annotators disagree

    resolution = row["resolution"]
    has_resolution = not (resolution is None or pd.isna(resolution) or str(resolution).strip() == "")

    if consensus is None:
        if not has_resolution:
            raise ValueError(f"{row['paper_id']}: disagreement {verdicts} without a resolution")
        key = str(resolution).strip().lower()
        if key not in RESOLUTION_MAP:
            raise ValueError(f"{row['paper_id']}: unrecognised resolution {resolution!r}")
        return RESOLUTION_MAP[key]

    if has_resolution:
        raise ValueError(f"{row['paper_id']}: consensus {verdicts} but resolution {resolution!r} is set")
    return consensus

### `majority_research_area`

Research-area columns are consolidated by majority vote.  
Blanks are ignored (annotators mostly left the field empty for excluded papers). 
On a tie the value asserted by the highest-ranked annotator wins, following the hierarchy **TC > ME > MC**. 
All blank stays blank.

For instance:
[TC: Misinformation, ME: Polarization, MC: Both] → "Misinformation"
[TC: blank, ME: Polarization, MC: Both] → "Polarization"
[TC: Misinformation, ME: Polarization, MC: Polarization] → "Polarization" 

In [18]:
def norm_research_area(value):
    """Map a research-area cell onto the canonical vocabulary; blank -> pd.NA."""
    if value is None or (not isinstance(value, str) and pd.isna(value)):
        return pd.NA
    key = str(value).strip().lower()
    if key == "":
        return pd.NA
    if key not in RESEARCH_AREA_CANON:
        raise ValueError(f"unrecognised research area: {value!r}")
    return RESEARCH_AREA_CANON[key]


def majority_research_area(row):
    """Majority vote over the annotators, ties broken by TC > ME > MC."""
    votes = {a: norm_research_area(row[f"{a}_Research area"]) for a in ANNOTATORS}
    votes = {a: v for a, v in votes.items() if v is not pd.NA}
    if not votes:
        return pd.NA

    counts = Counter(votes.values())
    top = max(counts.values())
    tied = {v for v, c in counts.items() if c == top}
    if len(tied) == 1:
        return next(iter(tied))
    for a in TIE_BREAK:
        if votes.get(a) in tied:
            return votes[a]

### `merge_topics` — merged keyword set

Annotators wrote comma-separated keyword lists. Each entry is split on commas, the stop phrases are
removed, surrounding whitespace and punctuation are stripped, empty fragments are dropped, and the
rest are deduplicated case-insensitively (keeping the first spelling seen, in MC, ME, TC order) and
joined with `"; "`.

`merge_notes` Notes are concatenated with an annotator label instead. For instance, "MC: This is a note; ME: Another note"

In [19]:
def _phrase_pattern(phrase):
    """Word-boundary-anchored, case-insensitive pattern (so "OT" never matches inside a word)."""
    left = r"\b" if phrase[:1].isalnum() else ""
    right = r"\b" if phrase[-1:].isalnum() else ""
    return re.compile(left + re.escape(phrase) + right, re.IGNORECASE)


PHRASE_RES = [_phrase_pattern(p) for p in REMOVE_PHRASES]


def clean_fragment(text):
    """Strip stop phrases and stray punctuation from a single comma-separated fragment."""
    for rx in PHRASE_RES:
        text = rx.sub(" ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip(" .;:,-\t\n")


def merge_topics(values):
    """Deduplicated keyword set from the topic cells of one paper, in annotator order."""
    keywords, seen = [], set()
    for value in values:
        if value is None or (not isinstance(value, str) and pd.isna(value)):
            continue
        for raw in str(value).split(","):
            fragment = clean_fragment(raw)
            if not fragment or fragment.lower() in seen:
                continue
            seen.add(fragment.lower())
            keywords.append(fragment)
    return "; ".join(keywords)


def merge_notes(notes_by_annotator):
    """Annotator-labelled concatenation of free-text notes, blanks skipped."""
    parts = []
    for annotator, value in notes_by_annotator.items():
        if value is None or (not isinstance(value, str) and pd.isna(value)):
            continue
        text = str(value).strip()
        if text:
            parts.append(f"{annotator}: {text}" if annotator else text)
    return " | ".join(parts)

## Consolidate each phase

Both builders return the same schema, so the phases can simply be concatenated. `to_discuss` is
`True` when at least one annotator flagged the paper, `False` otherwise (blanks count as
not flagged).

In [20]:
def consolidate_iaa(df, phase):
    """Consolidate a multi-annotator IAA file (round 1 or round 2)."""
    out = df[META_COLS].copy()
    out["include"] = df.apply(consolidate_include, axis=1).astype(bool)
    flags = pd.DataFrame({a: df[f"{a}_To discuss"].map(lambda v: to_bool(v) is True)
                          for a in ANNOTATORS})
    out["to_discuss"] = flags.any(axis=1)
    out["research_area"] = df.apply(majority_research_area, axis=1)
    out["topic"] = df.apply(lambda r: merge_topics([r[f"{a}_Topic/Ideas"] for a in ANNOTATORS]), axis=1)
    out["notes"] = df.apply(lambda r: merge_notes({a: r[f"{a}_Notes"] for a in ANNOTATORS}), axis=1)
    out["phase"] = phase
    return out[OUT_COLS]


def consolidate_single(df, phase):
    """Take the single annotator's answers as-is (remaining screening)."""
    out = df[META_COLS].copy()
    verdicts = df["Include"].map(to_bool)
    assert verdicts.notna().all(), "remaining screening has missing Include values"
    out["include"] = verdicts.astype(bool)
    out["to_discuss"] = df["To discuss"].map(lambda v: to_bool(v) is True)
    out["research_area"] = df["Research area"].map(norm_research_area)
    out["topic"] = df["Topic/Ideas"].map(lambda v: merge_topics([v]))
    out["notes"] = df["Notes"].map(lambda v: merge_notes({"": v}))
    out["phase"] = phase
    return out[OUT_COLS]


parts = [
    consolidate_iaa(round1, "round1"),
    consolidate_iaa(round2, "round2"),
    consolidate_single(remaining, "remaining"),
]
for part in parts:
    print(f"{part['phase'].iloc[0]:10} {len(part):>3} papers  "
          f"include={int(part['include'].sum()):>3}  exclude={int((~part['include']).sum()):>3}")

round1      50 papers  include= 23  exclude= 27
round2      20 papers  include= 10  exclude= 10
remaining  116 papers  include= 49  exclude= 67


## Concatenate and validate

In [21]:
consolidated = pd.concat(parts, ignore_index=True)
# Empty strings become blank cells in the spreadsheet.
consolidated[["topic", "notes"]] = consolidated[["topic", "notes"]].replace("", pd.NA)

assert len(consolidated) == 186, f"expected 186 rows, got {len(consolidated)}"
assert consolidated["paper_id"].is_unique, "duplicate paper_id"
assert consolidated["phase"].value_counts().to_dict() == {"remaining": 116, "round1": 50, "round2": 20}
assert consolidated["include"].dtype == bool and consolidated["include"].notna().all()
assert consolidated["to_discuss"].dtype == bool and consolidated["to_discuss"].notna().all()
areas = set(consolidated["research_area"].dropna())
assert areas <= set(RESEARCH_AREA_CANON.values()), f"unexpected research areas: {areas}"

print(f"Consolidated table: {consolidated.shape[0]} papers x {consolidated.shape[1]} columns")
consolidated.head()

Consolidated table: 186 papers x 14 columns


,paper_id,ids,year,sources,title,abstract,published_in,paper_type,include,to_discuss,research_area,topic,notes,phase
0,031fcedf9970f53c,scopus_eid: 2-s2.0-85115187550,2021,scopus,25th European Conference on Advances in Databa...,The proceedings contain 29 papers. The special...,scopus: Communications in Computer and Informa...,scopus: Conference Review,False,True,Misinformation,NaN,"MC: not a paper, it's the description of proce...",round1
1,0eea50f3d58ec619,"doi: 10.1109/bigdata55660.2022.10020460, scopu...",2022,scopus,Where did you tweet from? Inferring the origin...,Public conversations on Twitter comprise many ...,scopus: Proceedings 2022 IEEE International Co...,scopus: Conference Paper,False,True,NaN,NaN,MC: no misinformation or polarization | TC: OT...,round1
2,278d67b0621a5c36,"doi: 10.11114/smc.v14i1.7936, scopus_eid: 2-s2...",2026,"scopus, semantic_scholar",The Presence of Digital Volunteer and The Abse...,Extreme climate change around the world has di...,"scopus: Studies in Media and Communication, se...","scopus: Article, semantic_scholar: JournalArticle",False,True,NaN,NaN,ME: I am unable to see the misinfromation and ...,round1
3,2def0ea8fdc0f669,scopus_eid: 2-s2.0-105020071134,2026,scopus,18th International Conference on Social Comput...,The proceedings contain 23 papers. The special...,scopus: Lecture Notes in Computer Science,scopus: Conference Review,False,True,NaN,NaN,ME: This seems to be the intro to the entire p...,round1
4,3284d6b38a42e287,"arxiv_id: 2312.01217, semantic_scholar_id: 68b...",2023,semantic_scholar,Understanding Opinions Towards Climate Change ...,Social media platforms such as Twitter (now kn...,semantic_scholar: arXiv.org,semantic_scholar: JournalArticle,False,True,NaN,NaN,"MC: Not peer-reviewed. However, contains all t...",round1


Cross-check against the figures published in [03_iaa_analysis.ipynb](03_iaa_analysis.ipynb): round 1
has 10 unanimous includes plus 14 disagreements resolved to *include* (24), round 2 has 10 unanimous
includes and its single disagreement resolved to *not include* (10).

In [22]:
for df, phase in [(round1, "round1"), (round2, "round2")]:
    inc = pd.DataFrame({a: df[f"{a}_Include"].map(to_bool) for a in ANNOTATORS})
    n_seen = inc.map(lambda v: v is not pd.NA).sum(axis=1)
    n_yes = inc.map(lambda v: v is True).sum(axis=1)
    unanimous_include = int((n_yes == n_seen).sum())
    disagreement = (n_yes > 0) & (n_yes < n_seen)
    resolved_include = int(df.loc[disagreement, "resolution"].str.strip().str.lower().eq("include").sum())
    total = int(consolidated.loc[consolidated["phase"] == phase, "include"].sum())
    print(f"{phase}: {unanimous_include} unanimous include + {resolved_include} resolved to include "
          f"= {unanimous_include + resolved_include}  (consolidated: {total})")
    assert unanimous_include + resolved_include == total

round1: 10 unanimous include + 13 resolved to include = 23  (consolidated: 23)
round2: 10 unanimous include + 0 resolved to include = 10  (consolidated: 10)


In [23]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
consolidated.to_excel(OUTPUT_PATH, index=False)
print(f"Wrote {OUTPUT_PATH}")

Wrote ../data/processed/screening/screening_consolidated.xlsx


## Stop-phrase review

Every fragment of four or more words that survived into `topic`. Commentary that is still showing up
here belongs in `REMOVE_PHRASES` at the top of the notebook — add it and re-run.

In [24]:
surviving = sorted(
    {frag for cell in consolidated["topic"].dropna() for frag in cell.split("; ")
    #  if len(frag.split()) >= 4
    },
    key=str.lower,
)
print(f"{len(surviving)} fragments of >=4 words still in `topic`:")
for fragment in surviving:
    print("   ", fragment)

220 fragments of >=4 words still in `topic`:
    "social noise"
    about covid-19
    abstract retrieval
    affective computing
    affective polarization
    Affective polarization
    AI as disinformation weapon
    AI for qualitative methods
    AI influencers
    AI vs human influencers
    AI-generated images
    art installation
    Australia
    bayesian item response theory
    Believers and skeptics
    believers vs skeptics
    bias mitigation
    bias or unbiased classification
    bias vs unbiased posts
    Brazil
    categorisation of climate claims
    check worthiness detection
    Checkworthyness detection
    chinese social media
    claim detection
    claim object detection
    claim verification
    claimer detection
    climate change denial
    climate claims
    climate denial
    climate reports
    climate videos
    clustering
    cognitive bias
    comments network analysis
    community detection
    community etection
    computational argumentation
    C

## Summary statistics

In [25]:
summary = (consolidated
           .groupby("phase")
           .agg(papers=("paper_id", "size"),
                included=("include", "sum"),
                to_discuss=("to_discuss", "sum"),
                with_topic=("topic", "count"))
           .loc[["round1", "round2", "remaining"]])
summary["include_rate"] = (summary["included"] / summary["papers"] * 100).round(0).astype(int).astype(str) + "%"
print(summary.to_string())

n_included = int(consolidated["include"].sum())
print(f"\nTotal papers                      : {len(consolidated)}")
print(f"Papers passing screening          : {n_included} ({n_included / len(consolidated) * 100:.0f}%)")
print(f"Papers flagged to discuss         : {int(consolidated['to_discuss'].sum())}")

           papers  included  to_discuss  with_topic include_rate
phase                                                           
round1         50        23          10          27          46%
round2         20        10           3          13          50%
remaining     116        49           6          60          42%

Total papers                      : 186
Papers passing screening          : 82 (44%)
Papers flagged to discuss         : 19


In [26]:
included = consolidated[consolidated["include"]]
counts = included["research_area"].value_counts(dropna=False)
print("Research area among the included papers:")
for area, count in counts.items():
    label = "(blank)" if pd.isna(area) else area
    print(f"  {label:<16} {count:>3}  ({count / len(included) * 100:.1f}%)")

Research area among the included papers:
  Misinformation    34  (41.5%)
  Polarization      30  (36.6%)
  Both              18  (22.0%)
